# View 2: Latent Space Projections (UMAP)
Visualizes penultimate representation drift under corruption and isolates error clusters.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import umap

In [ ]:
CIFAR10_CLASSES = ["plane", "car", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

df = pd.read_parquet("../results/mobilenetv2_evaluation.parquet")
os.makedirs("../figures/phase1_human", exist_ok=True)

features = np.vstack(df["embedding"].values) # stack feature embeddings

reducer = umap.UMAP(n_neighbors=25, min_dist=0.15, random_state=42)
embeddings_2d = reducer.fit_transform(features)

df["umap_x"] = embeddings_2d[:, 0]
df["umap_y"] = embeddings_2d[:, 1]
df["class_name"] = df["true_label"].apply(lambda x: CIFAR10_CLASSES[x])

Plot Clean vs Maximum Corruption Latent Drift

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

df_clean = df[df["corruption"] == "clean"]
sns.scatterplot(
    data=df_clean,
    x="umap_x",
    y="umap_y",
    hue="class_name",
    style="cnn_correct",
    palette="tab10",
    alpha=0.8,
    ax=axes[0]
)
axes[0].set_title("UMAP Latent Space: Clean Baseline", fontsize=14)

df_sev5 = df[(df["corruption"] != "clean") & (df["severity"] == 5)]
sns.scatterplot(
    data=df_sev5,
    x="umap_x",
    y="umap_y",
    hue="corruption",
    style="cnn_correct",
    palette="Set2",
    alpha=0.7,
    ax=axes[1]
)
axes[1].set_title("UMAP Latent Space: Severity 5 Corruptions", fontsize=14)

plt.tight_layout()
plt.savefig("../figures/phase1_human/umap_representation_drift.png", dpi=300)
plt.show()